# CS781 Phase 1 - Selective SSM (Mamba-2 style) legal LM (1B-token, 2xT4, resumable)

**How this notebook runs**
- Use **Save Version -> Save & Run All (Commit)** with *GPU T4 x2*, so training keeps going even if the browser closes.
- Sections 4-5 (download + tokenizer + tokenization) run once. Tokenizer, token files and checkpoints are written to `/kaggle/working/model3_ssm_output`.
- Training (section 8) runs as a 2-GPU DDP job. After `SESSION_BUDGET_HOURS` it saves a checkpoint and stops cleanly; the evaluation cells then skip themselves.
- **To continue in a new session:** new version -> *Add Input -> Notebook Output Files -> this notebook* -> Run All. The tokenized corpus, tokenizer and newest checkpoint are found automatically (nothing is re-downloaded or re-tokenized) and the data order resumes exactly where it stopped.
- When the single pass over the corpus is finished, sections 9-10 evaluate the best checkpoint and build the submission zip.
- All artifacts of this notebook carry an `ssm` tag in their file names, so attaching the output of the decoder-only / encoder-decoder notebooks as well can never be mistaken for SSM files.

# 1. Login

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


# 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


# 3. Configuration

In [3]:
import os
import gc
import math
import time
import glob
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------------------------------------------------------
# Global Reproducibility & Device Configuration
# -----------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = torch.cuda.is_available()

# -----------------------------------------------------------------------------
# Hyperparameters: Model 3 (Selective SSM / Mamba)
# -----------------------------------------------------------------------------
TOTAL_TARGET_TOKENS = 1_000_000_000  # 1 Billion-token budget: ONE pass over it, LR anneals to MIN_LR exactly at the end
VAL_RATIO = 0.01

VOCAB_SIZE = 32000
D_MODEL = 768
N_LAYERS = 12
SSM_VARIANT = "mamba2"  # "mamba2": matmul-based chunked scan (SSD), fast in plain PyTorch.
                        # "mamba1": per-channel selective scan - only practical with custom CUDA kernels; in plain
                        #           PyTorch it ran at an ETA of ~438 h, so do not use it here.
D_STATE = 64          # SSM state size (use 16 for "mamba1")
EXPAND = 2            # Inner d_model expansion factor (d_inner = EXPAND * D_MODEL = 1536)
HEADDIM = 64          # "mamba2": channels per head -> 1536 / 64 = 24 heads, one scalar decay per head

TRAIN_BLOCK_SIZE = 2048      # Sequence length leveraging O(N) SSM efficiency

MICRO_BATCH_SIZE = 2         # samples per GPU per forward/backward. 4 ran out of memory on a 15 GB T4 (needs ~14.5 GB);
                             # if 2 still fails, set GRAD_CKPT = True (much lower memory, ~30% slower)
GLOBAL_BATCH_SAMPLES = 64    # samples per optimizer step across ALL GPUs; grad-accumulation is derived from it
LEARNING_RATE = 6e-4
MIN_LR = 6e-5
WEIGHT_DECAY = 0.1
WARMUP_FRACTION = 0.05       # warm-up = 5% of the total steps ...
MAX_WARMUP_STEPS = 1000      # ... but never more than this
# NOTE: MAX_TRAIN_STEPS is no longer hard-coded. One pass over the tokenized corpus =
# len(train_dataset) // GLOBAL_BATCH_SAMPLES; it is computed in section 7.

SCAN_CHUNK = 64              # chunk length of the chunked scan (see model_ssm.py)
GRAD_CKPT = False            # True = recompute each block in backward: less memory, ~30% slower

# Session / checkpoint policy (Kaggle sessions are time-limited)
SESSION_BUDGET_HOURS = 8.0   # training saves a checkpoint and stops cleanly this many hours after the notebook
                             # started (keep >= 1h below your session cap; raise it if your sessions allow ~12h)
CKPT_EVERY = 250             # optimizer steps between resumable checkpoints
EVAL_EVERY = 500             # optimizer steps between validation runs
LOG_EVERY = 50
VAL_SAMPLES = 400            # fixed validation subset (samples of 2,048 tokens)
DATALOADER_WORKERS = 2       # per GPU process
BYTES_PER_TOKEN = 4.0        # refined from real data after the tokenizer is trained

# Generation (settings of the course template: 128 WORDS, low temperature, nucleus sampling)
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384  # token budget used to reach 128 words; the prompt keeps the rest of the context
TEMPERATURE = 0.1
TOP_P = 0.95

# -----------------------------------------------------------------------------
# Output & Checkpoint Directories / artifact names
# -----------------------------------------------------------------------------
OUTPUT_DIR = "/kaggle/working/model3_ssm_output"
INPUT_ROOT = "/kaggle/input"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")

TOKENIZER_PATH = os.path.join(OUTPUT_DIR, "tokenizer_ssm.json")
TOKENIZATION_MARKER = os.path.join(OUTPUT_DIR, "ssm_tokenization_complete.json")
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model_ssm.pt")
CONFIG_PATH = os.path.join(OUTPUT_DIR, "train_config_ssm.json")
STATUS_PATH = os.path.join(OUTPUT_DIR, "train_status_ssm.json")

PREDICTION_PATH = os.path.join(OUTPUT_DIR, "predictions_model3.json")
METRICS_PATH = os.path.join(OUTPUT_DIR, "test_metrics_model3.json")
PARTIAL_PATH = PREDICTION_PATH + ".partial"
SUBMISSION_ZIP_PATH = "/kaggle/working/submission_model3.zip"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# Cross-session helpers
#    Kaggle wipes /kaggle/working between sessions. To continue in a new session, attach the previous
#    notebook version's OUTPUT as an input; artifacts are then found automatically (no hard-coded paths).
# -----------------------------------------------------------------------------
def find_artifact(pattern):
    """First match of `pattern` in this session's working dir, else anywhere under /kaggle/input."""
    for root in (OUTPUT_DIR, INPUT_ROOT):
        hits = sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True))
        if hits:
            return hits[0]
    return None

# Session clock (used for the clean-stop deadline of the training run)
try:
    import psutil
    SESSION_START = psutil.Process(os.getpid()).create_time()
except Exception:
    SESSION_START = time.time()

print(f"Model 3 config loaded | device: {device} | mixed precision (AMP): {use_amp}")
print(f"Output path: {OUTPUT_DIR}")

Model 3 config loaded | device: cuda | mixed precision (AMP): True
Output path: /kaggle/working/model3_ssm_output


# 4. File paths

In [4]:
# -----------------------------------------------------------------------------
# Hugging Face dataset download. The files go to the HF cache (outside /kaggle/working) so the multi-GB raw JSON
# is not saved with the notebook output. If a finished tokenized corpus + tokenizer exist (in /kaggle/working or in an
# attached previous-notebook output) the training files are not downloaded at all.
# -----------------------------------------------------------------------------
import os
import glob
import json
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

RELATIVE_TRAIN_FILES = [
    os.path.join("Phase_1_Pretraining", "Train", "train_supreme_court.json"),
    os.path.join("Phase_1_Pretraining", "Train", "train_high_court.json"),
    os.path.join("Phase_1_Pretraining", "Train", "train_statutes.json"),
    os.path.join("Phase_1_Pretraining", "Train", "train_tribunals.json"),
]
RELATIVE_TEST_FILE = os.path.join("Phase_1_Pretraining", "Test", "test.json")

_marker = find_artifact("ssm_tokenization_complete.json")
_prev_dir = os.path.dirname(_marker) if _marker else None
if (_prev_dir
        and os.path.exists(os.path.join(_prev_dir, "tokenizer_ssm.json"))
        and glob.glob(os.path.join(_prev_dir, "ssm_train_part_*.bin"))
        and glob.glob(os.path.join(_prev_dir, "ssm_val_part_*.bin"))):
    NEED_TOKENIZE = False
    TOKENIZED_DIR = _prev_dir
else:
    NEED_TOKENIZE = True
    TOKENIZED_DIR = OUTPUT_DIR

train_file_paths = []
if NEED_TOKENIZE:
    print(f"Downloading training files from Hugging Face Hub: '{HF_DATASET_REPO}'...")
    for rel_path in RELATIVE_TRAIN_FILES:
        try:
            downloaded_path = hf_hub_download(
                repo_id=HF_DATASET_REPO, filename=rel_path, repo_type="dataset", token=HF_TOKEN
            )
            train_file_paths.append(downloaded_path)
            size_mb = os.path.getsize(downloaded_path) / (1024 ** 2)
            print(f"  [+] Downloaded: {os.path.basename(rel_path)} ({size_mb:.2f} MB)")
        except Exception as e:
            print(f"  [!] Failed to download {rel_path}: {e}")
    if not train_file_paths:
        raise FileNotFoundError("Critical Error: no training JSON files could be downloaded!")
else:
    print(f"Found a finished tokenized corpus + tokenizer in {TOKENIZED_DIR} -> skipping the training-file download.")

print("\nDownloading test file...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, filename=RELATIVE_TEST_FILE, repo_type="dataset", token=HF_TOKEN
)
print(f"Test file: {test_file_path}")

if NEED_TOKENIZE:
    print(f"\nDiscovered {len(train_file_paths)} training file(s):")
    for path in train_file_paths:
        print(f"  - {path}")

Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

  [+] Downloaded: train_supreme_court.json (1261.30 MB)
  [!] Failed to download Phase_1_Pretraining/Train/train_high_court.json: 404 Client Error. (Request ID: Root=1-6ac38b52-64e0166065870264274bc271;3cfed0a5-df90-4c3f-9a78-9bc9c2484794)

Entry Not Found for url: https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_high_court.json.
  [!] Failed to download Phase_1_Pretraining/Train/train_statutes.json: 404 Client Error. (Request ID: Root=1-6ac38b52-519f3f85587bddac2a7cc471;70c17b4a-1004-401f-b3bf-e6f80ce7ab8b)

Entry Not Found for url: https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_statutes.json.


Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

  [+] Downloaded: train_tribunals.json (6183.59 MB)



test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Test file: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json

Discovered 2 training file(s):
  - /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json
  - /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json


# 5. BPE Tokeniser

In [5]:
!pip install -q ijson

import os
import gc
import ijson
import random
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]   # ids 0, 1, 2, 3  (<pad> = 0 is the loss ignore_index)
TARGET_VOCAB_SIZE = VOCAB_SIZE   # 32,000 (token ids must stay < 65,536 because the corpus is stored as uint16)

if not NEED_TOKENIZE:
    # ---- Reuse the tokenizer that produced the existing .bin files (must match them exactly) ----
    tokenizer = Tokenizer.from_file(os.path.join(TOKENIZED_DIR, "tokenizer_ssm.json"))
    print(f"Loaded existing tokenizer (vocab {tokenizer.get_vocab_size():,}) from {TOKENIZED_DIR}")
else:
    # -------------------------------------------------------------------------
    # Stream a representative sample across all files (ijson: one record at a time, constant memory;
    # the old json.load() of every multi-GB file could not even fit in RAM)
    # -------------------------------------------------------------------------
    SAMPLE_TARGET_BYTES = 200 * 1024 * 1024  # 200 MB text sample
    bytes_per_file = SAMPLE_TARGET_BYTES // max(len(train_file_paths), 1)

    sampled_texts = []
    total_bytes_collected = 0

    print(f"Streaming ~{SAMPLE_TARGET_BYTES / (1024 * 1024):.0f} MB of raw text for tokenizer training...")
    random.seed(SEED)

    for file_path in train_file_paths:
        file_bytes = 0
        file_name = os.path.basename(file_path)

        with open(file_path, "rb") as f:
            for record in ijson.items(f, "item"):
                text = record.get("text", "")
                if not text or len(text) < 30:
                    continue

                # Random sampling filter so we don't only take text from file head
                if random.random() < 0.1:
                    sampled_texts.append(text)
                    text_bytes = len(text.encode("utf-8"))
                    file_bytes += text_bytes
                    total_bytes_collected += text_bytes

                    if file_bytes >= bytes_per_file:
                        break

        gc.collect()
        print(f"  [+] Sampled {file_bytes / (1024 * 1024):.2f} MB from {file_name}")

    print(f"\nCorpus extraction complete: {len(sampled_texts):,} docs | {total_bytes_collected / (1024 * 1024):.2f} MB total.")

    # -------------------------------------------------------------------------
    # Byte-level BPE. initial_alphabet = all 256 byte symbols, so any character in the test set can be encoded
    # (without it, bytes that never occur in the sample would be silently dropped at inference time).
    # -------------------------------------------------------------------------
    print(f"\nTraining Byte-Level BPE tokenizer (target vocab size: {TARGET_VOCAB_SIZE:,})...")
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)

    trainer = trainers.BpeTrainer(
        vocab_size=TARGET_VOCAB_SIZE,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    tokenizer.train_from_iterator(sampled_texts, trainer=trainer)
    assert tokenizer.get_vocab_size() <= VOCAB_SIZE, "tokenizer vocab is larger than the model's embedding table"

    # Save it: the .bin files are only meaningful together with THIS tokenizer, and the notebook may restart.
    tokenizer.save(TOKENIZER_PATH)
    print(f"Saved tokenizer -> {TOKENIZER_PATH}  (vocab {tokenizer.get_vocab_size():,})")

    # Measure real bytes/token on a random probe so the corpus sub-sampling rate below is accurate.
    probe = random.sample(sampled_texts, min(2000, len(sampled_texts)))
    probe_bytes = sum(len(t.encode("utf-8")) for t in probe)
    probe_tokens = sum(len(e.ids) for e in tokenizer.encode_batch(probe))
    BYTES_PER_TOKEN = probe_bytes / max(probe_tokens, 1)
    print(f"Measured {BYTES_PER_TOKEN:.2f} bytes/token on a {len(probe):,}-document probe.")

    del sampled_texts, probe
    gc.collect()

# Special-token ids used for document framing in the corpus and at inference time
assert [tokenizer.token_to_id(t) for t in SPECIAL_TOKENS] == [0, 1, 2, 3], "special tokens must be ids 0..3"
PAD_ID, BOS_ID, EOS_ID = 0, 1, 2

print("-" * 70)
print(f"Vocabulary size : {tokenizer.get_vocab_size():,}")
print(f"Special tokens  : <pad>={PAD_ID} | <bos>={BOS_ID} | <eos>={EOS_ID} | <unk>=3")
sample_text = "The Supreme Court of India held that alltogether Section 302 applies to everyone."
encoded = tokenizer.encode(sample_text)
print(f"Test encoding   : {len(encoded.ids)} tokens -> '{tokenizer.decode(encoded.ids)}'")
print("-" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 5.0 MB/s eta 0:00:00
Streaming ~200 MB of raw text for tokenizer training...
  [+] Sampled 100.00 MB from train_supreme_court.json
  [+] Sampled 100.01 MB from train_tribunals.json

Corpus extraction complete: 16,003 docs | 200.02 MB total.

Training Byte-Level BPE tokenizer (target vocab size: 32,000)...



Saved tokenizer -> /kaggle/working/model3_ssm_output/tokenizer_ssm.json  (vocab 32,000)
Measured 4.54 bytes/token on a 2,000-document probe.
----------------------------------------------------------------------
Vocabulary size : 32,000
Special tokens  : <pad>=0 | <bos>=1 | <eos>=2 | <unk>=3
Test encoding   : 16 tokens -> 'The Supreme Court of India held that alltogether Section 302 applies to everyone.'
----------------------------------------------------------------------


# 6. Tokenization of datasets

In [6]:
import os
import glob
import json
import time
import shutil
import numpy as np
import ijson

os.environ["TOKENIZERS_PARALLELISM"] = "true"   # encode_batch is multi-threaded in Rust -> no multiprocessing needed

# -----------------------------------------------------------------------------
# Streaming tokenization.
# The old version json.load()ed the multi-GB files inside mp.Pool workers (~2x file size in RAM per worker);
# an OOM-killed worker makes Pool.starmap wait forever - that is where the notebook "halted". Here ONE process
# streams records with ijson (constant memory), tokenizes them in small batches with the multi-threaded Rust
# tokenizer, and writes the uint16 .bin files incrementally, with progress logging.
# Every document is framed as [BOS] + tokens + [EOS] (causal-LM framing, as before).
# -----------------------------------------------------------------------------
def tokenize_streaming(tokenizer, file_paths, out_dir, val_ratio=0.01, target_tokens=1_000_000_000,
                       keep_prob=1.0, docs_per_batch=1000, seed=42, log_every=20):
    os.makedirs(out_dir, exist_ok=True)

    # remove partial files from an earlier failed run (the Dataset globs them!)
    for p in glob.glob(os.path.join(out_dir, "ssm_train_part_*.bin")) + glob.glob(os.path.join(out_dir, "ssm_val_part_*.bin")):
        os.remove(p)

    rng = np.random.default_rng(seed)
    f_train = open(os.path.join(out_dir, "ssm_train_part_0.bin"), "wb")
    f_val = open(os.path.join(out_dir, "ssm_val_part_0.bin"), "wb")
    n_train = n_val = n_docs = n_batches = 0
    t0 = time.time()
    batch = []

    def flush():
        nonlocal n_train, n_val, n_docs, n_batches
        if not batch:
            return
        for enc in tokenizer.encode_batch(batch):
            ids = enc.ids
            if len(ids) < 10:
                continue
            arr = np.asarray([BOS_ID] + ids + [EOS_ID], dtype=np.uint16)
            if rng.random() < val_ratio:
                f_val.write(arr.tobytes()); n_val += arr.size
            else:
                f_train.write(arr.tobytes()); n_train += arr.size
        n_docs += len(batch)
        n_batches += 1
        batch.clear()
        if n_batches % log_every == 0:
            el = time.time() - t0
            tot = n_train + n_val
            print(f"[tok] {n_docs:,} docs | {tot/1e6:,.1f}M tokens | {tot/el/1e3:,.0f}k tok/s | {el/60:.1f} min", flush=True)

    try:
        done = False
        for path in file_paths:
            print(f"[tok] streaming {os.path.basename(path)}", flush=True)
            with open(path, "rb") as f:                       # binary mode for ijson
                for rec in ijson.items(f, "item"):            # one record at a time
                    text = rec.get("text")
                    if not text or len(text) < 30:
                        continue
                    # uniform random sub-sampling spreads the token budget over the WHOLE corpus
                    # (instead of only taking the first N GB of each file)
                    if keep_prob < 1.0 and rng.random() >= keep_prob:
                        continue
                    batch.append(text)
                    if len(batch) >= docs_per_batch:
                        flush()
                        if n_train + n_val >= target_tokens:
                            done = True
                            break
            if done:
                break
        flush()
    finally:
        f_train.close(); f_val.close()

    print(f"[tok] DONE  train={n_train:,}  val={n_val:,}  total={n_train+n_val:,} tokens in {(time.time()-t0)/60:.1f} min")
    return n_train, n_val, n_docs


if NEED_TOKENIZE:
    print(f"ijson backend: {ijson.backend}  (should be a yajl2_c variant; 'python' would be slow)")
    est_total_tokens = sum(os.path.getsize(p) for p in train_file_paths) / BYTES_PER_TOKEN
    KEEP_PROB = min(1.0, 1.08 * TOTAL_TARGET_TOKENS / est_total_tokens)   # ~8% head-room, stop at the target anyway
    print(f"Estimated corpus size: {est_total_tokens/1e9:.2f}B tokens -> keeping {KEEP_PROB:.1%} of documents "
          f"to reach a {TOTAL_TARGET_TOKENS/1e9:.1f}B-token budget.")

    n_train, n_val, n_docs = tokenize_streaming(
        tokenizer, train_file_paths, OUTPUT_DIR,
        val_ratio=VAL_RATIO, target_tokens=TOTAL_TARGET_TOKENS, keep_prob=KEEP_PROB, seed=SEED,
    )
    with open(TOKENIZATION_MARKER, "w") as f:   # written LAST: its presence means "tokenization finished"
        json.dump({"train_tokens": n_train, "val_tokens": n_val, "docs": n_docs, "vocab_size": tokenizer.get_vocab_size(),
                   "bytes_per_token": BYTES_PER_TOKEN, "keep_prob": KEEP_PROB}, f)
    TOKENIZED_DIR = OUTPUT_DIR
else:
    with open(os.path.join(TOKENIZED_DIR, "ssm_tokenization_complete.json")) as f:
        _m = json.load(f)
    print(f"Reusing tokenized corpus from {TOKENIZED_DIR}: {_m['train_tokens']:,} train / {_m['val_tokens']:,} val tokens.")
    if os.path.abspath(TOKENIZED_DIR) != os.path.abspath(OUTPUT_DIR):
        # copy forward so THIS version's output is self-contained (the next session then only needs this one attached)
        for pat in ("ssm_train_part_*.bin", "ssm_val_part_*.bin", "tokenizer_ssm.json", "ssm_tokenization_complete.json"):
            for src in glob.glob(os.path.join(TOKENIZED_DIR, pat)):
                dst = os.path.join(OUTPUT_DIR, os.path.basename(src))
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
        TOKENIZED_DIR = OUTPUT_DIR
        print("Copied tokenized corpus + tokenizer into /kaggle/working so they persist with this version.")

ijson backend: yajl2_c  (should be a yajl2_c variant; 'python' would be slow)
Estimated corpus size: 1.72B tokens -> keeping 62.8% of documents to reach a 1.0B-token budget.
[tok] streaming train_supreme_court.json
[tok] 20,000 docs | 93.7M tokens | 840k tok/s | 1.9 min
[tok] streaming train_tribunals.json
[tok] 40,000 docs | 185.8M tokens | 842k tok/s | 3.7 min
[tok] 60,000 docs | 225.1M tokens | 859k tok/s | 4.4 min
[tok] 80,000 docs | 260.3M tokens | 873k tok/s | 5.0 min
[tok] 100,000 docs | 310.4M tokens | 878k tok/s | 5.9 min
[tok] 120,000 docs | 349.6M tokens | 884k tok/s | 6.6 min
[tok] 140,000 docs | 389.8M tokens | 888k tok/s | 7.3 min
[tok] 160,000 docs | 405.7M tokens | 891k tok/s | 7.6 min
[tok] 180,000 docs | 420.8M tokens | 894k tok/s | 7.8 min
[tok] 200,000 docs | 435.8M tokens | 898k tok/s | 8.1 min
[tok] 220,000 docs | 450.2M tokens | 901k tok/s | 8.3 min
[tok] 240,000 docs | 475.9M tokens | 906k tok/s | 8.8 min
[tok] 260,000 docs | 504.0M tokens | 913k tok/s | 9.2 min

# 7. Model

In [7]:
%%writefile /kaggle/working/model3_ssm_output/model_ssm.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        var = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(var + self.eps) * self.weight


def selective_scan(dt, A, B, C, x, chunk: int = 64):
    """
    h_t = exp(dt_t * A) * h_{t-1} + dt_t * B_t * x_t ;  y_t = sum_n C_t[n] * h_t[:, n]

    Same maths as a step-by-step loop over all L positions, but done in two levels so that
    only `chunk + L/chunk` python-level steps are needed instead of L (2,048 -> ~96):
      1. inside every chunk, run the recurrence from h=0, all chunks in parallel;
      2. a short sequential pass over the chunks produces the state entering each chunk;
      3. add that carried state's contribution, exp(cumsum(dt*A)) * carry, to every position.
    Every exponent is <= 0 (A < 0, dt >= 0), so nothing can overflow.
    All inputs must be float32.  dt, x: (b, L, d)   A: (d, n)   B, C: (b, L, n)  ->  y: (b, L, d)
    """
    b, L, d = x.shape
    n = A.shape[1]
    pad = (-L) % chunk
    if pad:  # dt = 0 -> decay 1 and no input: the state is left untouched by the padded steps
        dt = F.pad(dt, (0, 0, 0, pad))
        x = F.pad(x, (0, 0, 0, pad))
        B = F.pad(B, (0, 0, 0, pad))
        C = F.pad(C, (0, 0, 0, pad))
    nc = (L + pad) // chunk
    dt = dt.reshape(b, nc, chunk, d)
    x = x.reshape(b, nc, chunk, d)
    B = B.reshape(b, nc, chunk, n)
    C = C.reshape(b, nc, chunk, n)

    dtA = dt.unsqueeze(-1) * A                                   # (b, nc, T, d, n)
    dA = torch.exp(dtA)
    dBx = dt.unsqueeze(-1) * x.unsqueeze(-1) * B.unsqueeze(3)    # (b, nc, T, d, n)

    # 1. local scan inside every chunk (starting from zero state)
    h = x.new_zeros(b, nc, d, n)
    ys = []
    for t in range(chunk):
        h = dA[:, :, t] * h + dBx[:, :, t]
        ys.append((h * C[:, :, t].unsqueeze(2)).sum(-1))         # (b, nc, d)
    y_local = torch.stack(ys, dim=2)                             # (b, nc, T, d)

    # 2. state entering each chunk
    S = torch.cumsum(dtA, dim=2)                                 # log of the decay accumulated inside the chunk
    P_end = torch.exp(S[:, :, -1])                               # (b, nc, d, n)
    carry = x.new_zeros(b, d, n)
    carries = []
    for c in range(nc):
        carries.append(carry)
        carry = P_end[:, c] * carry + h[:, c]
    carry_in = torch.stack(carries, dim=1)                       # (b, nc, d, n)

    # 3. contribution of the carried state to every position of its chunk
    y_corr = (torch.exp(S) * carry_in.unsqueeze(2) * C.unsqueeze(3)).sum(-1)   # (b, nc, T, d)

    y = (y_local + y_corr).reshape(b, nc * chunk, d)
    return y[:, :L]


class SelectiveSSM(nn.Module):
    """
    Selective State-Space Module (Mamba core): input-dependent (Delta, B, C) and a linear recurrence over positions.
    """
    def __init__(self, d_model: int = 768, d_state: int = 16, expand: int = 2,
                 dt_rank: int = None, kernel_size: int = 4, scan_chunk: int = 64):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = d_model * expand
        self.dt_rank = dt_rank if dt_rank is not None else math.ceil(d_model / 16)
        self.scan_chunk = scan_chunk

        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=kernel_size,
                                padding=kernel_size - 1, groups=self.d_inner, bias=True)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.reset_dt_parameters()

    def reset_dt_parameters(self):
        """Mamba's timescale init: dt in [1e-3, 1e-1] (log-uniform), stored through the inverse softplus."""
        dt_init_std = self.dt_rank ** -0.5
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        dt_min, dt_max = 1e-3, 1e-1
        dt = torch.exp(torch.rand(self.d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min))
        dt = dt.clamp(min=1e-4)
        inv_softplus_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_softplus_dt)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, _ = x.shape

        xz = self.in_proj(x)
        x_branch, z_branch = xz.chunk(2, dim=-1)

        x_conv = x_branch.transpose(1, 2)
        x_conv = self.conv1d(x_conv)[:, :, :seq_len]       # drop the right-hand causal padding
        x_act = F.silu(x_conv.transpose(1, 2))

        x_ssm = self.x_proj(x_act)
        dt_raw, B, C = torch.split(x_ssm, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt_raw))

        # The recurrence always runs in float32 (exp/cumsum/products are not safe in fp16 autocast).
        with torch.autocast(device_type=x.device.type, enabled=False):
            A = -torch.exp(self.A_log.float())
            x32 = x_act.float()
            y = selective_scan(dt.float(), A, B.float(), C.float(), x32, chunk=self.scan_chunk)
            y = y + x32 * self.D
            y = y * F.silu(z_branch.float())
        return self.out_proj(y.to(z_branch.dtype))


def segsum(x: torch.Tensor) -> torch.Tensor:
    """(..., T) -> (..., T, T) with out[i, j] = sum(x[j+1..i]) for i >= j and -inf above the diagonal."""
    T = x.size(-1)
    c = torch.cumsum(x, dim=-1)
    out = c.unsqueeze(-1) - c.unsqueeze(-2)
    return out.masked_fill(~torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device)), float("-inf"))


def ssd_scan(x, dt, A, B, C, chunk: int = 64):
    """
    Chunked state-space-duality scan (Mamba-2): same recurrence as a sequential loop,
        h_t = exp(A * dt_t) * h_{t-1} + dt_t * (x_t outer B_t) ;   y_t = h_t . C_t
    but computed with a handful of batched matrix multiplications instead of L (or L * d_inner * d_state) tiny steps:
      1. inside every chunk: (C B^T) * decay-mask  ->  matmul with x          (the diagonal blocks)
      2. the state each chunk leaves behind                                   (matmul)
      3. a short recurrence over the chunk states                             (tiny)
      4. the carried state's contribution to every position of its chunk      (matmul)
    All exponents are <= 0 (A < 0, dt >= 0), so nothing can overflow. Inputs must be float32.
    x: (b, L, h, p)   dt: (b, L, h)   A: (h,)   B, C: (b, L, n) shared by all heads   ->   y: (b, L, h, p)
    """
    b, L, h, p = x.shape
    n = B.size(-1)
    pad = (-L) % chunk
    if pad:  # dt = 0 -> decay 1 and no input: padded steps leave the state untouched
        x = F.pad(x, (0, 0, 0, 0, 0, pad))
        dt = F.pad(dt, (0, 0, 0, pad))
        B = F.pad(B, (0, 0, 0, pad))
        C = F.pad(C, (0, 0, 0, pad))
    nc = (L + pad) // chunk

    X = (x * dt.unsqueeze(-1)).reshape(b, nc, chunk, h, p)
    Bc = B.reshape(b, nc, chunk, n)
    Cc = C.reshape(b, nc, chunk, n)
    Ad = (A.view(1, 1, h) * dt).reshape(b, nc, chunk, h).permute(0, 3, 1, 2)          # (b, h, nc, chunk)
    Acs = torch.cumsum(Ad, dim=-1)

    # 1. diagonal blocks
    Lm = torch.exp(segsum(Ad))                                                         # (b, h, nc, l, s)
    G = torch.einsum("bcln,bcsn->bcls", Cc, Bc)
    Y_diag = torch.einsum("bhcls,bcshp->bclhp", G.unsqueeze(1) * Lm, X)

    # 2. state left behind by each chunk
    decay_states = torch.exp(Acs[..., -1:] - Acs)                                      # (b, h, nc, l)
    states = torch.einsum("bcln,bclhp->bchpn", Bc, X * decay_states.permute(0, 2, 3, 1).unsqueeze(-1))

    # 3. recurrence over chunk states (state entering chunk c = decayed sum of all earlier chunk states)
    states = torch.cat([torch.zeros_like(states[:, :1]), states], dim=1)               # (b, nc+1, h, p, n)
    decay_chunk = torch.exp(segsum(F.pad(Acs[..., -1], (1, 0))))                       # (b, h, nc+1, nc+1)
    states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)[:, :-1]            # (b, nc, h, p, n)

    # 4. carried state -> outputs
    Y_off = torch.einsum("bcln,bchpn->bclhp", Cc, states) * torch.exp(Acs).permute(0, 2, 3, 1).unsqueeze(-1)

    return (Y_diag + Y_off).reshape(b, nc * chunk, h, p)[:, :L]


class SSDMixer(nn.Module):
    """
    Mamba-2 style selective state-space mixer: input-dependent (dt, B, C), a scalar decay A per head, heads of size
    `headdim`, B and C shared by all heads, gated RMS-norm before the output projection.
    Its scan is matmul-based (see ssd_scan), which is what makes it practical to train without custom CUDA kernels.
    """
    def __init__(self, d_model: int = 768, d_state: int = 64, expand: int = 2, headdim: int = 64,
                 kernel_size: int = 4, scan_chunk: int = 64):
        super().__init__()
        self.d_inner = d_model * expand
        assert self.d_inner % headdim == 0, "d_model * expand must be divisible by headdim"
        self.d_state = d_state
        self.headdim = headdim
        self.nheads = self.d_inner // headdim
        self.scan_chunk = scan_chunk
        conv_dim = self.d_inner + 2 * d_state                       # x, B, C go through the causal conv together

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner + 2 * d_state + self.nheads, bias=False)   # z, x, B, C, dt
        self.conv1d = nn.Conv1d(conv_dim, conv_dim, kernel_size=kernel_size, padding=kernel_size - 1,
                                groups=conv_dim, bias=True)
        self.dt_bias = nn.Parameter(torch.zeros(self.nheads))
        self.A_log = nn.Parameter(torch.zeros(self.nheads))
        self.D = nn.Parameter(torch.ones(self.nheads))
        self.norm = RMSNorm(self.d_inner)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.reset_dt_parameters()

    def reset_dt_parameters(self):
        """dt in [1e-3, 1e-1] (log-uniform) through the inverse softplus; A in [1, 16] (log-uniform in magnitude)."""
        dt_min, dt_max = 1e-3, 1e-1
        dt = torch.exp(torch.rand(self.nheads) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        with torch.no_grad():
            self.dt_bias.copy_(dt + torch.log(-torch.expm1(-dt)))
            self.A_log.copy_(torch.log(torch.empty(self.nheads).uniform_(1.0, 16.0)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, L, _ = x.shape
        z, xBC, dt = torch.split(self.in_proj(x), [self.d_inner, self.d_inner + 2 * self.d_state, self.nheads], dim=-1)
        xBC = F.silu(self.conv1d(xBC.transpose(1, 2))[:, :, :L].transpose(1, 2))     # drop the right-hand causal padding
        xs, Bm, Cm = torch.split(xBC, [self.d_inner, self.d_state, self.d_state], dim=-1)

        # The scan always runs in float32 (exp / cumsum / products are not safe in fp16 autocast).
        with torch.autocast(device_type=x.device.type, enabled=False):
            dt = F.softplus(dt.float() + self.dt_bias)
            A = -torch.exp(self.A_log.float())
            xh = xs.float().reshape(b, L, self.nheads, self.headdim)
            y = ssd_scan(xh, dt, A, Bm.float(), Cm.float(), chunk=self.scan_chunk)
            y = (y + xh * self.D.view(1, 1, -1, 1)).reshape(b, L, self.d_inner)
            y = self.norm(y * F.silu(z.float()))
        return self.out_proj(y.to(z.dtype))


class MambaBlock(nn.Module):
    """Residual block with RMS normalisation around either mixer ("mamba2" = SSDMixer, "mamba1" = SelectiveSSM)."""
    def __init__(self, d_model: int, d_state: int = 64, expand: int = 2, scan_chunk: int = 64,
                 variant: str = "mamba2", headdim: int = 64):
        super().__init__()
        self.norm = RMSNorm(d_model)
        if variant == "mamba2":
            self.ssm = SSDMixer(d_model=d_model, d_state=d_state, expand=expand, headdim=headdim, scan_chunk=scan_chunk)
        elif variant == "mamba1":
            self.ssm = SelectiveSSM(d_model=d_model, d_state=d_state, expand=expand, scan_chunk=scan_chunk)
        else:
            raise ValueError(f"unknown SSM variant: {variant}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.ssm(self.norm(x))


class Model3_SSM(nn.Module):
    """Selective state-space causal language model."""
    def __init__(self, vocab_size: int = 32000, d_model: int = 768, n_layers: int = 12,
                 d_state: int = 64, expand: int = 2, scan_chunk: int = 64, grad_ckpt: bool = False,
                 variant: str = "mamba2", headdim: int = 64):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.grad_ckpt = grad_ckpt

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            MambaBlock(d_model=d_model, d_state=d_state, expand=expand, scan_chunk=scan_chunk,
                       variant=variant, headdim=headdim)
            for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight      # weight tying

        self.apply(self._init_weights)
        # self.apply() above also hits the mamba1 dt_proj Linear (weights -> N(0, 0.02), bias -> 0) and would wipe
        # the timescale init: put it back (the mamba2 mixer keeps dt_bias / A_log as plain parameters, harmless there).
        for m in self.modules():
            if isinstance(m, (SelectiveSSM, SSDMixer)):
                m.reset_dt_parameters()

    @staticmethod
    def _init_weights(module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, input_ids: torch.Tensor, targets: torch.Tensor = None, last_only: bool = False):
        x = self.embedding(input_ids)
        for layer in self.layers:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                # recompute the block in backward: the (b, L, d_inner, d_state) scan tensors are never kept for all layers
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        x = self.norm_f(x)
        if last_only:                    # generation: only the last position's logits are needed
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1), ignore_index=0)  # 0 = <pad>
        return logits, loss

Writing /kaggle/working/model3_ssm_output/model_ssm.py


In [8]:
import sys
import importlib
sys.path.insert(0, OUTPUT_DIR)
import model_ssm
importlib.reload(model_ssm)
from model_ssm import Model3_SSM, selective_scan, ssd_scan

MODEL_KWARGS = dict(vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS, d_state=D_STATE, expand=EXPAND,
                    scan_chunk=SCAN_CHUNK, grad_ckpt=GRAD_CKPT, variant=SSM_VARIANT, headdim=HEADDIM)

# ---- 1. each fast scan must reproduce the plain step-by-step recurrence (float64, CPU, tiny shapes) ----
def _naive_scan(dt, A, B, C, x):                       # mamba1 recurrence
    b, L, d = x.shape
    n = A.shape[1]
    h = x.new_zeros(b, d, n)
    ys = []
    for t in range(L):
        h = torch.exp(dt[:, t].unsqueeze(-1) * A) * h + dt[:, t].unsqueeze(-1) * B[:, t].unsqueeze(1) * x[:, t].unsqueeze(-1)
        ys.append((h * C[:, t].unsqueeze(1)).sum(-1))
    return torch.stack(ys, dim=1)

def _naive_ssd(x, dt, A, B, C):                        # mamba2 recurrence, one scalar decay per head
    b, L, h, p = x.shape
    n = B.size(-1)
    st = x.new_zeros(b, h, p, n)
    ys = []
    for t in range(L):
        dec = torch.exp(A.view(1, h) * dt[:, t]).view(b, h, 1, 1)
        st = dec * st + (dt[:, t].unsqueeze(-1) * x[:, t]).unsqueeze(-1) * B[:, t].view(b, 1, 1, n)
        ys.append((st * C[:, t].view(b, 1, 1, n)).sum(-1))
    return torch.stack(ys, dim=1)

_g = torch.Generator().manual_seed(0)
_f64 = dict(generator=_g, dtype=torch.float64)
_b, _L = 2, 70                                           # L = 70 is not a multiple of the chunk -> exercises the padding path
for _scale in (2.0, 40.0):                               # 40 = extreme timescales (decay exponents of hundreds per step)
    _d, _n = 6, 4
    _dt = torch.rand(_b, _L, _d, **_f64) * _scale
    _A = -(torch.rand(_d, _n, **_f64) + 0.1)
    _B, _C = torch.randn(_b, _L, _n, **_f64), torch.randn(_b, _L, _n, **_f64)
    _x = torch.randn(_b, _L, _d, **_f64)
    _out = selective_scan(_dt, _A, _B, _C, _x, chunk=16)
    _err1 = ((_out - _naive_scan(_dt, _A, _B, _C, _x)).abs().max() / _out.abs().max()).item()

    _h, _p, _n2 = 3, 4, 5
    _dt2 = torch.rand(_b, _L, _h, **_f64) * _scale
    _A2 = -(torch.rand(_h, **_f64) + 0.1)
    _B2, _C2 = torch.randn(_b, _L, _n2, **_f64), torch.randn(_b, _L, _n2, **_f64)
    _x2 = torch.randn(_b, _L, _h, _p, **_f64)
    _out2 = ssd_scan(_x2, _dt2, _A2, _B2, _C2, chunk=16)
    _err2 = ((_out2 - _naive_ssd(_x2, _dt2, _A2, _B2, _C2)).abs().max() / _out2.abs().max()).item()
    assert torch.isfinite(_out).all() and torch.isfinite(_out2).all()
    assert _err1 < 1e-9 and _err2 < 1e-9, f"chunked scan deviates from the sequential recurrence: {_err1}, {_err2}"
    print(f"scan check (timescale x{_scale:g}): mamba1 rel.err {_err1:.1e} | mamba2/SSD rel.err {_err2:.1e}")

# ---- 2. size report + forward-pass check on the CPU (the GPUs are not touched; training builds the real model per GPU) ----
_probe = Model3_SSM(**MODEL_KWARGS)
n_params = sum(p.numel() for p in _probe.parameters())
_mix = _probe.layers[0].ssm
_dt0 = F.softplus(_mix.dt_bias if hasattr(_mix, "dt_bias") else _mix.dt_proj.bias)
print("=" * 70)
print("MODEL 3 ARCHITECTURE SUMMARY (selective SSM)")
print("=" * 70)
print(f"variant {SSM_VARIANT} | d_model {D_MODEL} | d_inner {D_MODEL * EXPAND} | d_state {D_STATE} | layers {N_LAYERS} | vocab {VOCAB_SIZE:,}"
      + (f" | heads {_mix.nheads} x {HEADDIM}" if hasattr(_mix, "nheads") else ""))
print(f"Parameters: {n_params / 1e6:.1f}M  (tied embedding / output head)")
print(f"Initial dt range: [{_dt0.min().item():.4f}, {_dt0.max().item():.4f}]  (expected ~[0.001, 0.1])")
_ids = torch.randint(4, VOCAB_SIZE, (1, 70))
with torch.no_grad():
    _logits, _loss = _probe(_ids, _ids)
print(f"Forward check | input {tuple(_ids.shape)} -> logits {tuple(_logits.shape)} | loss {_loss.item():.3f} "
      f"(random-init reference ~ ln(vocab) = {math.log(VOCAB_SIZE):.2f})")
print("=" * 70)
del _probe, _logits, _loss
gc.collect()

scan check (timescale x2): mamba1 rel.err 1.7e-16 | mamba2/SSD rel.err 2.2e-16
scan check (timescale x40): mamba1 rel.err 2.0e-16 | mamba2/SSD rel.err 6.1e-16
MODEL 3 ARCHITECTURE SUMMARY (selective SSM)
variant mamba2 | d_model 768 | d_inner 1536 | d_state 64 | layers 12 | vocab 32,000 | heads 24 x 64
Parameters: 68.6M  (tied embedding / output head)
Initial dt range: [0.0011, 0.0920]  (expected ~[0.001, 0.1])
Forward check | input (1, 70) -> logits (1, 70, 32000) | loss 10.429 (random-init reference ~ ln(vocab) = 10.37)


67

# 8. Dataloaders

In [9]:
%%writefile /kaggle/working/model3_ssm_output/data_ssm.py
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset


class CausalMemmapDataset(Dataset):
    """
    Disk-backed uint16 token stream cut into NON-overlapping blocks.
    Sample i = tokens [i*block, i*block + block + 1) -> (x, y) = (first `block`, last `block`), so the last target of a
    block is the first input of the next one and no token is skipped.
    (The old dataset used a stride of 1: len() was ~num_tokens, i.e. billions of near-identical windows.)
    """
    def __init__(self, file_pattern: str, block_size: int):
        self.block_size = block_size
        self.bin_files = [f for f in sorted(glob.glob(file_pattern)) if os.path.getsize(f) > 0]
        self.mmaps = [np.memmap(f, dtype=np.uint16, mode="r") for f in self.bin_files]
        self.lengths = [len(m) for m in self.mmaps]
        self.total_tokens = sum(self.lengths)
        self.cum_lengths = np.cumsum([0] + self.lengths)
        self.num_samples = max(0, (self.total_tokens - 1) // block_size)

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int):
        need = self.block_size + 1
        pos = int(idx) * self.block_size
        file_idx = int(np.searchsorted(self.cum_lengths, pos, side="right") - 1)
        local = pos - int(self.cum_lengths[file_idx])
        parts = []
        while need > 0 and file_idx < len(self.mmaps):          # a window may span several partition files
            piece = self.mmaps[file_idx][local:local + need]
            parts.append(np.asarray(piece, dtype=np.int64))
            need -= len(piece)
            file_idx += 1
            local = 0
        chunk = np.concatenate(parts)
        return torch.from_numpy(chunk[:-1]), torch.from_numpy(chunk[1:])


def rank_micro_batches(num_samples, world_size, rank, micro_bs, accum, seed, start_step=0, max_steps=None):
    """
    Deterministic, resumable data order (one pass, no repeats).
    Optimizer step s always consumes perm[s*G:(s+1)*G] with G = world*micro_bs*accum, so a resumed run
    (even on a different number of GPUs, if G is kept constant) continues exactly where it stopped.
    Returns (total_steps, list_of_index_lists) for this rank, starting at `start_step`.
    """
    G = world_size * micro_bs * accum
    total_steps = num_samples // G
    n_steps = total_steps if max_steps is None else min(total_steps, max_steps)
    perm = np.random.default_rng(seed).permutation(num_samples)[: total_steps * G]
    perm = perm.reshape(total_steps, accum, world_size, micro_bs)[:n_steps]
    mine = perm[start_step:, :, rank, :].reshape(-1, micro_bs)
    return total_steps, mine.tolist()


def val_micro_batches(num_samples, world_size, rank, micro_bs, n_total):
    """Fixed random subset of the validation set, split across ranks."""
    per = world_size * micro_bs
    n = (min(num_samples, n_total) // per) * per
    if n == 0:
        return []
    idx = np.random.default_rng(0).permutation(num_samples)[:n].reshape(-1, world_size, micro_bs)
    return idx[:, rank, :].tolist()

Writing /kaggle/working/model3_ssm_output/data_ssm.py


In [10]:
import sys
sys.path.insert(0, OUTPUT_DIR)
import importlib
import data_ssm
importlib.reload(data_ssm)
from data_ssm import CausalMemmapDataset

train_dataset = CausalMemmapDataset(os.path.join(TOKENIZED_DIR, "ssm_train_part_*.bin"), TRAIN_BLOCK_SIZE)
val_dataset = CausalMemmapDataset(os.path.join(TOKENIZED_DIR, "ssm_val_part_*.bin"), TRAIN_BLOCK_SIZE)

# One pass over the whole tokenized corpus; the cosine schedule is built on this number, so the learning rate
# anneals to MIN_LR exactly when the data runs out.
MAX_TRAIN_STEPS = len(train_dataset) // GLOBAL_BATCH_SAMPLES
WARMUP_STEPS = int(max(10, min(MAX_WARMUP_STEPS, WARMUP_FRACTION * MAX_TRAIN_STEPS)))
TOKENS_PER_STEP = GLOBAL_BATCH_SAMPLES * TRAIN_BLOCK_SIZE

print("=" * 70)
print(f"Dataset loaded: {len(train_dataset):,} training samples | {len(val_dataset):,} val samples (non-overlapping {TRAIN_BLOCK_SIZE}-token blocks).")
print(f"Train tokens: {train_dataset.total_tokens/1e6:,.1f}M  |  tokens/optimizer step: {TOKENS_PER_STEP:,}")
print(f"MAX_TRAIN_STEPS = {MAX_TRAIN_STEPS:,} (one full pass)  |  warm-up = {WARMUP_STEPS} steps")
print("Wall-clock is printed during training (s/step and ETA) - check it over the first ~50 steps before committing to the full pass.")
print("=" * 70)

Dataset loaded: 483,542 training samples | 5,015 val samples (non-overlapping 2048-token blocks).
Train tokens: 990.3M  |  tokens/optimizer step: 131,072
MAX_TRAIN_STEPS = 7,555 (one full pass)  |  warm-up = 377 steps
Wall-clock is printed during training (s/step and ETA) - check it over the first ~50 steps before committing to the full pass.


# 9. Training Loop

Training script (written to disk and launched with `torchrun` on both T4s; resumable, time-boxed).

In [11]:
%%writefile /kaggle/working/model3_ssm_output/train_ddp.py
"""
Resumable DDP pre-training script for the selective-SSM (Mamba) legal LM.
Launched by the notebook with:  python -m torch.distributed.run --standalone --nproc_per_node=<GPUs> train_ddp.py config.json
Works with 1 or 2 GPUs (global batch is fixed, grad-accumulation adapts) and on CPU (gloo) for smoke tests.
"""
import os, sys, json, time, math, glob, contextlib
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader

cfg = json.load(open(sys.argv[1]))
sys.path.insert(0, cfg["output_dir"])
from model_ssm import Model3_SSM
from data_ssm import CausalMemmapDataset, rank_micro_batches, val_micro_batches

# ----------------------------------------------------------------------------- distributed setup
rank = int(os.environ.get("RANK", 0))
world = int(os.environ.get("WORLD_SIZE", 1))
local_rank = int(os.environ.get("LOCAL_RANK", 0))
use_cuda = torch.cuda.is_available()
if use_cuda:
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
else:
    device = torch.device("cpu")
if world > 1:
    dist.init_process_group(backend="nccl" if use_cuda else "gloo")
is_main = rank == 0
use_amp = use_cuda


def log(*a):
    if is_main:
        print(*a, flush=True)


torch.manual_seed(cfg["seed"])
np.random.seed(cfg["seed"])

# ----------------------------------------------------------------------------- hyper-parameters
micro_bs = cfg["micro_batch_size"]
gbs = cfg["global_batch_samples"]
assert gbs % (world * micro_bs) == 0, f"global batch {gbs} not divisible by world({world})*micro({micro_bs})"
accum = gbs // (world * micro_bs)
block = cfg["block_size"]
tokens_per_step = gbs * block

train_ds = CausalMemmapDataset(os.path.join(cfg["tokenized_dir"], "ssm_train_part_*.bin"), block)
val_ds = CausalMemmapDataset(os.path.join(cfg["tokenized_dir"], "ssm_val_part_*.bin"), block)
total_steps = len(train_ds) // gbs
max_steps = total_steps if cfg.get("max_train_steps") is None else min(total_steps, int(cfg["max_train_steps"]))
warmup = int(max(10, min(cfg["max_warmup_steps"], cfg["warmup_fraction"] * max_steps)))
lr_max, lr_min = cfg["learning_rate"], cfg["min_lr"]


def get_lr(step):
    if step < warmup:
        return lr_max * (step + 1) / warmup
    if step >= max_steps:
        return lr_min
    ratio = (step - warmup) / max(1, max_steps - warmup)
    return lr_min + 0.5 * (1.0 + math.cos(math.pi * ratio)) * (lr_max - lr_min)


ckpt_dir = cfg["checkpoint_dir"]
os.makedirs(ckpt_dir, exist_ok=True)
latest_path = os.path.join(ckpt_dir, "latest_checkpoint_ssm.pt")
best_path = os.path.join(ckpt_dir, "best_model_ssm.pt")
status_path = cfg["status_path"]

log(f"[setup] world={world} device={device} micro_bs={micro_bs} accum={accum} global_batch={gbs} samples "
    f"({tokens_per_step:,} tokens/step)")
log(f"[setup] train samples={len(train_ds):,} val samples={len(val_ds):,} -> {max_steps:,} optimizer steps "
    f"(one pass), warmup={warmup}")

# ----------------------------------------------------------------------------- model / optimizer
model = Model3_SSM(**cfg["model"]).to(device)
raw_model = model


def configure_optimizer(m, weight_decay, learning_rate):
    """No weight decay on biases, 1-D tensors (norm gains, D) and A_log."""
    decay, no_decay = [], []
    for name, p in m.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() < 2 or name.endswith("bias") or "norm" in name or "A_log" in name or name.endswith(".D"):
            no_decay.append(p)
        else:
            decay.append(p)
    groups = [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8)


opt = configure_optimizer(raw_model, cfg["weight_decay"], lr_max)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


def pick_resume():
    """Newest checkpoint (highest step) in the working dir or in any attached input (previous notebook output)."""
    cands = []
    for root in cfg["resume_roots"]:
        cands += glob.glob(os.path.join(root, "**", "latest_checkpoint_ssm.pt"), recursive=True)
    best_p, best_s = None, -1
    for p in sorted(set(cands)):
        try:
            try:
                m = torch.load(p, map_location="cpu", mmap=True, weights_only=False)
            except TypeError:
                m = torch.load(p, map_location="cpu")
            s = int(m.get("step", -1))
            del m
        except Exception as e:  # corrupt / half-written file
            log(f"[resume] skipping unreadable checkpoint {p}: {e}")
            continue
        if s > best_s:
            best_p, best_s = p, s
    return best_p, best_s


global_step = 0
best_val = float("inf")
resume_path, resume_step = pick_resume()
if resume_path is not None:
    ck = torch.load(resume_path, map_location="cpu", weights_only=False)
    raw_model.load_state_dict(ck["model_state"])
    opt.load_state_dict(ck["optimizer_state"])
    if use_amp and ck.get("scaler_state"):
        scaler.load_state_dict(ck["scaler_state"])
    global_step = int(ck["step"])
    best_val = float(ck.get("best_val_loss", float("inf")))
    log(f"[resume] restored step {global_step:,} from {resume_path} (best val so far: {best_val:.4f})")
    del ck
else:
    log("[resume] no checkpoint found -> starting from scratch")

if world > 1:
    model = DDP(model, device_ids=[local_rank] if use_cuda else None)
n_params = sum(p.numel() for p in raw_model.parameters())
log(f"[model] {n_params/1e6:.1f}M parameters")


# ----------------------------------------------------------------------------- helpers
def write_status(finished):
    if not is_main:
        return
    tmp = status_path + ".tmp"
    with open(tmp, "w") as f:
        json.dump({"finished": bool(finished), "step": global_step, "max_steps": max_steps,
                   "best_val_loss": None if best_val == float("inf") else best_val,
                   "updated": time.time()}, f)
    os.replace(tmp, status_path)


def save_checkpoint():
    if not is_main:
        return
    ck = {"model_state": raw_model.state_dict(), "optimizer_state": opt.state_dict(),
          "scaler_state": scaler.state_dict() if use_amp else None,
          "step": global_step, "best_val_loss": best_val, "tokens_seen": global_step * tokens_per_step}
    tmp = latest_path + ".tmp"
    torch.save(ck, tmp)
    os.replace(tmp, latest_path)
    log(f"[ckpt] saved step {global_step:,} -> {latest_path}")


@torch.no_grad()
def evaluate():
    raw_model.eval()
    batches = val_micro_batches(len(val_ds), world, rank, micro_bs, cfg["val_samples"])
    tot = torch.zeros(2, device=device)
    if batches:
        for x, y in DataLoader(val_ds, batch_sampler=batches, num_workers=0):
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                _, l = raw_model(x, y)
            tot[0] += l.float()
            tot[1] += 1
    if world > 1:
        dist.all_reduce(tot)
    raw_model.train()
    return (tot[0] / tot[1]).item() if tot[1] > 0 else float("nan")


# ----------------------------------------------------------------------------- training loop
t_start = time.time()
if global_step >= max_steps:
    log("[train] checkpoint is already at the final step - nothing left to train.")
    write_status(True)
else:
    _, train_batches = rank_micro_batches(len(train_ds), world, rank, micro_bs, accum, cfg["seed"],
                                          start_step=global_step, max_steps=max_steps)
    loader = DataLoader(train_ds, batch_sampler=train_batches, num_workers=cfg["num_workers"], pin_memory=use_cuda)
    it = iter(loader)
    model.train()
    opt.zero_grad(set_to_none=True)
    t_last, steps_since_log = time.time(), 0
    finished = False
    log(f"[train] starting at step {global_step:,}/{max_steps:,}")

    while global_step < max_steps:
        lr = get_lr(global_step)
        for g in opt.param_groups:
            g["lr"] = lr

        loss_sum = torch.zeros((), device=device)
        for micro in range(accum):
            x, y = next(it)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            sync_ctx = contextlib.nullcontext() if (world == 1 or micro == accum - 1) else model.no_sync()
            with sync_ctx:
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                    _, loss = model(x, y)
                scaler.scale(loss / accum).backward()
            loss_sum += loss.detach().float()
            del loss, _

        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        global_step += 1
        steps_since_log += 1

        if global_step % cfg["log_every"] == 0 or global_step == 1:
            avg = loss_sum / accum
            if world > 1:
                dist.all_reduce(avg)
                avg = avg / world
            v = avg.item()
            if not math.isfinite(v):
                log(f"[train] NON-FINITE loss at step {global_step} - aborting (last good checkpoint is kept).")
                sys.exit(2)
            now = time.time()
            sps = (now - t_last) / max(1, steps_since_log)
            eta_h = sps * (max_steps - global_step) / 3600
            mem = f" | mem {torch.cuda.max_memory_allocated()/2**30:.1f}GB" if use_cuda else ""
            log(f"step {global_step:6d}/{max_steps} | loss {v:.4f} | lr {lr:.2e} | gnorm {gnorm.item():.2f} | "
                f"{sps:.2f}s/step | {tokens_per_step/sps/1e3:.1f}k tok/s | elapsed {(now-t_start)/3600:.2f}h | "
                f"ETA {eta_h:.2f}h{mem}")
            t_last, steps_since_log = now, 0

        stop = False
        if global_step % 10 == 0:
            flag = torch.tensor([1.0 if time.time() > cfg["deadline_epoch"] else 0.0], device=device)
            if world > 1:
                dist.all_reduce(flag, op=dist.ReduceOp.MAX)
            stop = flag.item() > 0

        do_eval = (global_step % cfg["eval_every"] == 0) or global_step == max_steps
        if do_eval:
            val = evaluate()
            ppl = math.exp(val) if val < 20 else float("inf")
            log(f"\n[val] step {global_step} | val loss {val:.4f} | ppl {ppl:.2f}\n")
            if is_main and val == val and val < best_val:  # val == val -> not NaN
                best_val = val
                tmp = best_path + ".tmp"
                torch.save({"model_state": raw_model.state_dict(), "step": global_step, "best_val_loss": best_val}, tmp)
                os.replace(tmp, best_path)
                log(f"[val] new best -> {best_path}")
            if world > 1:  # all ranks must agree on best_val for the checkpoint they might write
                bv = torch.tensor([best_val if best_val != float("inf") else 1e30], device=device)
                dist.broadcast(bv, src=0)
                best_val = float(bv.item()) if bv.item() < 1e29 else float("inf")

        if do_eval or stop or global_step % cfg["ckpt_every"] == 0:
            save_checkpoint()
        if global_step == max_steps:
            finished = True
        if stop and not finished:
            log(f"[train] session time budget reached at step {global_step:,} - checkpoint saved, stopping cleanly.")
            break

    write_status(finished)
    log(f"[train] {'FINISHED' if finished else 'PAUSED'} at step {global_step:,}/{max_steps:,} | best val {best_val:.4f}")

if world > 1:
    dist.barrier()
    dist.destroy_process_group()

Writing /kaggle/working/model3_ssm_output/train_ddp.py


In [12]:
import os
import sys
import json
import time
import subprocess

NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS >= 1, "No GPU found - enable 'GPU T4 x2' under Session options -> Accelerator."
assert GLOBAL_BATCH_SAMPLES % (NUM_GPUS * MICRO_BATCH_SIZE) == 0, \
    f"GLOBAL_BATCH_SAMPLES ({GLOBAL_BATCH_SAMPLES}) must be divisible by NUM_GPUS*MICRO_BATCH_SIZE ({NUM_GPUS * MICRO_BATCH_SIZE})"

DEADLINE_EPOCH = SESSION_START + SESSION_BUDGET_HOURS * 3600
remaining_h = (DEADLINE_EPOCH - time.time()) / 3600
print(f"{NUM_GPUS} GPU(s) | grad-accum per GPU = {GLOBAL_BATCH_SAMPLES // (NUM_GPUS * MICRO_BATCH_SIZE)} | "
      f"time left before the clean-stop deadline: {remaining_h:.1f} h")
if remaining_h < 0.25:
    print("WARNING: almost no time left in this session's budget - training will pause right away. "
          "Start a new session / raise SESSION_BUDGET_HOURS.")

train_cfg = dict(
    seed=SEED, output_dir=OUTPUT_DIR, tokenized_dir=TOKENIZED_DIR, checkpoint_dir=CHECKPOINT_DIR,
    status_path=STATUS_PATH, resume_roots=[OUTPUT_DIR, INPUT_ROOT],
    micro_batch_size=MICRO_BATCH_SIZE, global_batch_samples=GLOBAL_BATCH_SAMPLES, block_size=TRAIN_BLOCK_SIZE,
    learning_rate=LEARNING_RATE, min_lr=MIN_LR, weight_decay=WEIGHT_DECAY,
    warmup_fraction=WARMUP_FRACTION, max_warmup_steps=MAX_WARMUP_STEPS,
    max_train_steps=None,                      # None -> one full pass over the tokenized corpus
    num_workers=DATALOADER_WORKERS, log_every=LOG_EVERY, eval_every=EVAL_EVERY, ckpt_every=CKPT_EVERY,
    val_samples=VAL_SAMPLES, deadline_epoch=DEADLINE_EPOCH, model=MODEL_KWARGS,
)
with open(CONFIG_PATH, "w") as f:
    json.dump(train_cfg, f, indent=2)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
# env["NCCL_P2P_DISABLE"] = "1"   # uncomment ONLY if the run hangs before step 1 (NCCL peer-to-peer issue)

cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={NUM_GPUS}",
       os.path.join(OUTPUT_DIR, "train_ddp.py"), CONFIG_PATH]
print("Launching:", " ".join(cmd), "\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Training process exited with code {rc} - see the log above. "
                       f"The last good checkpoint (if any) is in {CHECKPOINT_DIR}.")

status = json.load(open(STATUS_PATH)) if os.path.exists(STATUS_PATH) else {}
TRAINING_DONE = bool(status.get("finished", False))
print("\nTRAINING FINISHED - run the evaluation cells below." if TRAINING_DONE else
      f"\nPAUSED at step {status.get('step')}/{status.get('max_steps')}: checkpoint saved. Evaluation is skipped; "
      "start the next session (attach this notebook's output as input) to continue.")

2 GPU(s) | grad-accum per GPU = 16 | time left before the clean-stop deadline: 7.7 h
Launching: /usr/bin/python3 -m torch.distributed.run --standalone --nproc_per_node=2 /kaggle/working/model3_ssm_output/train_ddp.py /kaggle/working/model3_ssm_output/train_config_ssm.json 


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[setup] world=2 device=cuda:0 micro_bs=2 accum=16 global_batch=64 samples (131,072 tokens/step)
[setup] train samples=483,542 val samples=5,015 -> 7,555 optimizer steps (one pass), warmup=377
[resume] no checkpoint found -> starting from scratch
[model] 68.6M parameters
[train] starting at step 0/7,555
step      1/7555 | loss 10.5218 | lr 1.59e-06 | gnorm 1.41 | 10.76s/step | 12.2k tok/s | elapsed 0.00h | ETA 22.58h | mem 7.4G

# 10. Test Evaluation and zip submission file

In [13]:
import sys
sys.path.insert(0, OUTPUT_DIR)
import os
import json
import time
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer
from model_ssm import Model3_SSM

print("=" * 70)
print("INFERENCE & SUBMISSION GENERATION")
print("=" * 70)

# The tokenizer lives in memory only if the tokenizer cell ran in this kernel; otherwise load the saved one.
if "tokenizer" not in globals():
    tokenizer = Tokenizer.from_file(find_artifact("tokenizer_ssm.json"))
BOS_ID, EOS_ID = tokenizer.token_to_id("<bos>"), tokenizer.token_to_id("<eos>")

# -----------------------------------------------------------------------------
# Only evaluate once pre-training has actually finished (a paused run just skips this cell)
# -----------------------------------------------------------------------------
_status = json.load(open(STATUS_PATH)) if os.path.exists(STATUS_PATH) else {}
TRAINING_DONE = bool(_status.get("finished", False))

if not TRAINING_DONE:
    print(f"Pre-training is not finished yet (step {_status.get('step')}/{_status.get('max_steps')}). "
          "Skipping evaluation - continue training in the next session (attach this notebook's output as input).")
else:
    # -------------------------------------------------------------------------
    # Load best weights (fall back to the latest checkpoint if the best-model file is not in this session's output)
    # -------------------------------------------------------------------------
    _best = BEST_MODEL_PATH if os.path.exists(BEST_MODEL_PATH) else (
        find_artifact("best_model_ssm.pt") or find_artifact("latest_checkpoint_ssm.pt"))
    assert _best is not None, "No trained weights found."
    ckpt = torch.load(_best, map_location="cpu", weights_only=False)
    eval_model = Model3_SSM(**MODEL_KWARGS).to(device)
    eval_model.load_state_dict(ckpt["model_state"])
    eval_model.eval()
    print(f"Loaded weights from {_best} (step {ckpt.get('step', 0)})")
    del ckpt

    # -------------------------------------------------------------------------
    # Test file: the course template stores it as {id: {"text": prompt}}; a plain list of records also works.
    # -------------------------------------------------------------------------
    def load_test_items(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            return [(str(k), v.get("text", "") if isinstance(v, dict) else str(v)) for k, v in data.items()]
        return [(str(r.get("id", i)), r.get("text", r.get("prompt", ""))) for i, r in enumerate(data)]

    test_items = load_test_items(test_file_path)
    print(f"Loaded {len(test_items):,} test prompts.")

    # -------------------------------------------------------------------------
    # Test perplexity on the prompt texts (first TRAIN_BLOCK_SIZE tokens of every record).
    #   all tokens : every token is scored given everything before it (standard causal-LM perplexity)
    #   2nd half   : only the positions the encoder-decoder notebook scores (tokens[half+1:]), for a like-for-like comparison
    # -------------------------------------------------------------------------
    @torch.no_grad()
    def evaluate_test_ppl(model, items):
        tot_loss = tot_tok = half_loss = half_tok = 0.0
        for _, text in items:
            if not text:
                continue
            ids = tokenizer.encode(text).ids
            if len(ids) < 4:
                continue
            ids = ids[:TRAIN_BLOCK_SIZE]
            seq = [BOS_ID] + ids
            x = torch.tensor([seq[:-1]], dtype=torch.long, device=device)
            y = torch.tensor([seq[1:]], dtype=torch.long, device=device)     # y[j] == ids[j]
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                logits, _ = model(x)
            tok_loss = F.cross_entropy(logits[0].float(), y[0], reduction="none")
            tot_loss += tok_loss.sum().item(); tot_tok += tok_loss.numel()
            second = tok_loss[len(ids) // 2 + 1:]
            half_loss += second.sum().item(); half_tok += second.numel()

        def to_ppl(l, n):
            if n == 0:
                return float("inf"), float("inf")
            avg = l / n
            return avg, (math.exp(avg) if avg < 20 else float("inf"))
        return to_ppl(tot_loss, tot_tok), to_ppl(half_loss, half_tok)

    (test_loss, test_ppl), (half_loss, half_ppl) = evaluate_test_ppl(eval_model, test_items)
    print("\n" + "=" * 50)
    print(f"  Model 3 Test Loss / PPL (all tokens)        : {test_loss:.4f} / {test_ppl:.2f}")
    print(f"  Model 3 Test Loss / PPL (2nd-half positions): {half_loss:.4f} / {half_ppl:.2f}")
    print("=" * 50 + "\n")
    with open(METRICS_PATH, "w") as f:
        json.dump({"test_loss": test_loss, "test_ppl": test_ppl,
                   "test_loss_second_half": half_loss, "test_ppl_second_half": half_ppl}, f, indent=2)

    # -------------------------------------------------------------------------
    # Generation: temperature + top-p sampling, stops once TARGET_GENERATION_WORDS words exist (<eos> is blocked so the
    # text cannot end early), and the
    # output is cut to exactly that many words (the task asks for 128 WORDS, not tokens).
    # The prompt keeps its END if it is too long (the continuation has to follow the last words of the prompt).
    # The whole context is re-run for every new token, so this is the slow part of the notebook.
    # -------------------------------------------------------------------------
    MAX_PROMPT_TOKENS = TRAIN_BLOCK_SIZE - MAX_GENERATION_TOKENS - 1

    def sample_next(logits, temperature, top_p):
        probs = F.softmax(logits.float() / max(temperature, 1e-5), dim=-1)
        sorted_p, sorted_i = torch.sort(probs, descending=True, dim=-1)
        drop = (sorted_p.cumsum(-1) - sorted_p) > top_p      # smallest set whose mass reaches top_p (top token always kept)
        sorted_p = sorted_p.masked_fill(drop, 0.0)
        sorted_p = sorted_p / sorted_p.sum(-1, keepdim=True)
        return sorted_i.gather(-1, torch.multinomial(sorted_p, 1))

    @torch.no_grad()
    def generate_completion(model, prompt):
        model.eval()
        seq = [BOS_ID] + tokenizer.encode(prompt).ids[-MAX_PROMPT_TOKENS:]
        prompt_len = len(seq)
        for step in range(MAX_GENERATION_TOKENS):
            x = torch.tensor([seq[-TRAIN_BLOCK_SIZE:]], dtype=torch.long, device=device)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                logits, _ = model(x, last_only=True)
            step_logits = logits[:, -1, :].float()
            step_logits[:, :4] = float("-inf")     # <pad>/<bos>/<eos>/<unk> are never generated: always produce the full 128 words
            nxt = sample_next(step_logits, TEMPERATURE, TOP_P).item()
            seq.append(nxt)
            if (step + 1) % 16 == 0 and len(tokenizer.decode(seq[prompt_len:]).split()) >= TARGET_GENERATION_WORDS:
                break
        return " ".join(tokenizer.decode(seq[prompt_len:]).split()[:TARGET_GENERATION_WORDS])

    # Partial results are saved every 25 prompts; a new session picks them up and only generates what is missing.
    predictions = {}
    _partial = find_artifact(os.path.basename(PARTIAL_PATH))
    if _partial:
        with open(_partial, "r", encoding="utf-8") as f:
            predictions = json.load(f)
        print(f"Resuming generation: {len(predictions):,} predictions already done ({_partial}).")

    todo = [(i, t) for i, t in test_items if i not in predictions]
    torch.manual_seed(SEED)          # sampled completions are reproducible
    print(f"Generating completions for {len(todo):,} test prompts...")
    t0 = time.time()
    for n, (doc_id, prompt) in enumerate(todo, 1):
        predictions[doc_id] = {"text": generate_completion(eval_model, prompt)}
        if n <= 2:
            print(f"\n--- Sample {n} ---\nPrompt   : {prompt[:150]}...\nGenerated: {predictions[doc_id]['text'][:200]}")
        if n % 25 == 0 or n == len(todo):
            el = time.time() - t0
            print(f"  {len(predictions):,}/{len(test_items):,} done | {el / n:.1f} s/prompt | ETA {(len(todo) - n) * el / n / 60:.0f} min", flush=True)
            with open(PARTIAL_PATH, "w", encoding="utf-8") as f:
                json.dump(predictions, f, ensure_ascii=False)

    ordered = {i: predictions[i] for i, _ in test_items}          # same order as the test file
    with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
        json.dump(ordered, f, indent=2, ensure_ascii=False)
    print(f"\nEvaluation complete! Saved {len(ordered):,} predictions -> {PREDICTION_PATH}")

INFERENCE & SUBMISSION GENERATION
Pre-training is not finished yet (step 3690/7555). Skipping evaluation - continue training in the next session (attach this notebook's output as input).


In [14]:
import zipfile

# -----------------------------------------------------------------------------
# Submission packaging: zip ONLY the prediction file (stored as prediction.json inside the archive)
# -----------------------------------------------------------------------------
if os.path.exists(PREDICTION_PATH):
    print(f"Zipping prediction file from: {PREDICTION_PATH}...")
    with zipfile.ZipFile(SUBMISSION_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(PREDICTION_PATH, arcname="prediction.json")

    zip_size_mb = os.path.getsize(SUBMISSION_ZIP_PATH) / (1024 ** 2)
    print("\n" + "=" * 50)
    print("Model 3 Submission Package Complete!")
    print(f"  - Archive Path : {SUBMISSION_ZIP_PATH}")
    print(f"  - Included File: prediction.json")
    print(f"  - Archive Size : {zip_size_mb:.2f} MB")
    print("=" * 50)
else:
    print(f"No prediction file at {PREDICTION_PATH} yet - it is written by the evaluation cell once pre-training has finished.")

No prediction file at /kaggle/working/model3_ssm_output/predictions_model3.json yet - it is written by the evaluation cell once pre-training has finished.


### -----------------------------------COMPLETE-------------------------------------